In [1]:
!pip install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
from pathlib import Path
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()
uploaded = files.upload()


Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


In [3]:
def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")
extra_readings_df = load_csv("meter_readings_extra_12000.csv")
meter_df = load_csv("meter_master.csv")
substation_df = load_csv("substation_master.csv")
tariff_df = load_csv("tariff_master.csv")
maintenance_df = load_csv("maintenance_tickets_1600.csv")


In [4]:
for name, df in [("readings_df", readings_df), ("extra_readings_df", extra_readings_df),
                  ("meter_df", meter_df), ("substation_df", substation_df),
                  ("tariff_df", tariff_df), ("maintenance_df", maintenance_df)]:
    print(name, df.count())
    df.printSchema()


readings_df 90000
root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

extra_readings_df 12000
root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |--

### 1 - Load inspection

In [5]:
readings_df.printSchema()
print("Rows:", readings_df.count())
readings_df.show(10, truncate=False)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Rows: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0000001 |M0082   |2026-04-01 0

### 2 - Select columns and rename source_type separately

In [6]:
exercise_02_df = readings_df.select("reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status")
renamed_readings_df = readings_df.withColumnRenamed("source_type", "collection_mode")
exercise_02_df.show(10, truncate=False)
renamed_readings_df.select("reading_id", "collection_mode").show(10, truncate=False)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
|RD0000001 |M0082   |2026-04-01 02:00:00|22.53     |Valid         |
|RD0000002 |M0020   |2026-01-31 06:00:00|2.97      |Valid         |
|RD0000003 |M0054   |2026-04-11 10:15:00|8.61      |Estimated     |
|RD0000004 |M0111   |2026-03-14 00:00:00|1.87      |Valid         |
|RD0000005 |M0135   |2026-05-27 01:45:00|4.39      |Valid         |
|RD0000006 |M0139   |2026-02-20 02:15:00|0.36      |Valid         |
|RD0000007 |M0142   |2026-02-01 05:30:00|79.47     |Valid         |
|RD0000008 |M0126   |2026-03-01 20:45:00|28.35     |Rejected      |
|RD0000009 |M0005   |2026-05-03 10:30:00|1.96      |Estimated     |
|RD0000010 |M0052   |2026-04-10 05:45:00|22.52     |Valid         |
+----------+--------+-------------------+----------+--------------+
only showing top 10 rows
+----------+-----------

### 3 — Valid readings above 20 kWh, sorted descending

In [7]:
exercise_03_df = readings_df.filter((F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20)).orderBy(F.col("energy_kwh").desc())
exercise_03_df.show(20, truncate=False)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0041703 |M0101   |2026-03-06 22:30:00|110.0     |213.1    |0.822       |0             |Valid         |SmartMeter |31.1                 |OP010      |
|RD0077494 |M0014   |2026-05-05 17:30:00|109.99    |NULL     |0.771       |0             |Valid         |Manual     |39.0                 |OP031      |
|RD0034412 |M0026   |2026-02-24 21:15:00|109.96    |231.6    |0.964       |0             |Valid         |SmartMeter |25.3                 |OP017      |
|RD0039063 |M0101   |2026-02-01 00:45:00|109.96    |231.6    |0.835       |0            

### 4 - Create energy_band

In [8]:
exercise_04_df = readings_df.withColumn("energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium").otherwise("High"))
exercise_04_df.select("reading_id", "energy_kwh", "energy_band").show(20, truncate=False)

+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
|RD0000001 |22.53     |Medium     |
|RD0000002 |2.97      |Low        |
|RD0000003 |8.61      |Medium     |
|RD0000004 |1.87      |Low        |
|RD0000005 |4.39      |Low        |
|RD0000006 |0.36      |Low        |
|RD0000007 |79.47     |High       |
|RD0000008 |28.35     |High       |
|RD0000009 |1.96      |Low        |
|RD0000010 |22.52     |Medium     |
|RD0000011 |10.51     |Medium     |
|RD0000012 |1.49      |Low        |
|RD0000013 |15.75     |Medium     |
|RD0000014 |15.96     |Medium     |
|RD0000015 |6.06      |Medium     |
|RD0000016 |25.82     |High       |
|RD0000017 |6.23      |Medium     |
|RD0000018 |3.66      |Low        |
|RD0000019 |104.8     |High       |
|RD0000020 |1.35      |Low        |
+----------+----------+-----------+
only showing top 20 rows


### 5 - Null counts

In [9]:
exercise_05_df = readings_df.agg(
    F.sum(F.col("energy_kwh").isNull().cast("long")).alias("null_energy_kwh"),
    F.sum(F.col("voltage_v").isNull().cast("long")).alias("null_voltage_v"),
    F.sum(F.col("power_factor").isNull().cast("long")).alias("null_power_factor"),
    F.sum(F.col("operator_id").isNull().cast("long")).alias("null_operator_id"))
exercise_05_df.show()

+---------------+--------------+-----------------+----------------+
|null_energy_kwh|null_voltage_v|null_power_factor|null_operator_id|
+---------------+--------------+-----------------+----------------+
|           1620|          2070|             1440|            5400|
+---------------+--------------+-----------------+----------------+



### 6 — Fill null operator and power factor in a new DataFrame

In [10]:
exercise_06_df = readings_df.na.fill({"operator_id": "UNASSIGNED", "power_factor": 0.90})
exercise_06_df.select("reading_id", "operator_id", "power_factor").show(20, truncate=False)

+----------+-----------+------------+
|reading_id|operator_id|power_factor|
+----------+-----------+------------+
|RD0000001 |OP029      |0.939       |
|RD0000002 |OP025      |0.845       |
|RD0000003 |OP007      |0.879       |
|RD0000004 |UNASSIGNED |0.831       |
|RD0000005 |OP019      |0.966       |
|RD0000006 |OP021      |0.788       |
|RD0000007 |OP016      |0.986       |
|RD0000008 |OP025      |0.818       |
|RD0000009 |OP010      |0.817       |
|RD0000010 |OP028      |0.804       |
|RD0000011 |OP030      |0.9         |
|RD0000012 |OP027      |0.81        |
|RD0000013 |OP005      |0.807       |
|RD0000014 |OP019      |0.884       |
|RD0000015 |OP036      |0.919       |
|RD0000016 |OP035      |0.895       |
|RD0000017 |OP003      |0.762       |
|RD0000018 |OP021      |0.914       |
|RD0000019 |OP014      |0.953       |
|RD0000020 |OP020      |0.771       |
+----------+-----------+------------+
only showing top 20 rows


### 7 — Drop missing energy or voltage and report removed rows

In [11]:
exercise_07_df = readings_df.na.drop(subset=["energy_kwh", "voltage_v"])
print("Before:", readings_df.count(), "After:", exercise_07_df.count(), "Removed:", readings_df.count() - exercise_07_df.count())

Before: 90000 After: 86349 Removed: 3651


### 8 — Valid energy by site_type

In [13]:
readings_meter_df = readings_df.join(meter_df, "meter_id", "left")
exercise_08_df = readings_meter_df.filter(F.col("reading_status") == "Valid").groupBy("site_type").agg(F.sum("energy_kwh").alias("total_valid_energy_kwh")).orderBy(F.desc("total_valid_energy_kwh"))
exercise_08_df.show(truncate=False)

+---------------+----------------------+
|site_type      |total_valid_energy_kwh|
+---------------+----------------------+
|Industrial     |710317.4100000021     |
|Commercial     |451525.1799999996     |
|Public Facility|138870.41             |
|Residential    |100863.2900000004     |
+---------------+----------------------+



### 9 — Valid energy by region and city

In [14]:
readings_meter_substation_df = readings_meter_df.join(substation_df, "substation_id", "left")
exercise_09_df = readings_meter_substation_df.filter(F.col("reading_status") == "Valid").groupBy("region", "city").agg(F.sum("energy_kwh").alias("total_valid_energy_kwh")).orderBy("region", "city")
exercise_09_df.show(truncate=False)

+-------+-------------+----------------------+
|region |city         |total_valid_energy_kwh|
+-------+-------------+----------------------+
|Central|Indore       |176503.91000000012    |
|North  |Delhi        |94234.89000000014     |
|North  |Jaipur       |47863.15999999996     |
|North  |Lucknow      |142610.3999999998     |
|South  |Bengaluru    |73787.59999999998     |
|South  |Chennai      |51829.05999999996     |
|South  |Hyderabad    |51937.789999999884    |
|South  |Kochi        |204272.4600000002     |
|South  |Visakhapatnam|205015.82999999932    |
|West   |Ahmedabad    |138627.7400000004     |
|West   |Mumbai       |52116.20999999999     |
|West   |Pune         |162777.23999999996    |
+-------+-------------+----------------------+



### 10 — Per-reading energy charge using tariff rate

In [15]:
readings_tariff_df = readings_meter_df.join(tariff_df, "tariff_id", "left")
exercise_10_df = readings_tariff_df.filter((F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull()).withColumn("energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr"))
exercise_10_df.select("reading_id", "energy_kwh", "rate_per_kwh_inr", "energy_charge_inr").show(20, truncate=False)

+----------+----------+----------------+------------------+
|reading_id|energy_kwh|rate_per_kwh_inr|energy_charge_inr |
+----------+----------+----------------+------------------+
|RD0000001 |22.53     |6.7             |150.95100000000002|
|RD0000002 |2.97      |6.4             |19.008000000000003|
|RD0000003 |8.61      |8.2             |70.60199999999999 |
|RD0000004 |1.87      |5.1             |9.536999999999999 |
|RD0000005 |4.39      |8.2             |35.998            |
|RD0000006 |0.36      |6.4             |2.304             |
|RD0000007 |79.47     |9.1             |723.1769999999999 |
|RD0000009 |1.96      |5.1             |9.995999999999999 |
|RD0000010 |22.52     |6.7             |150.88400000000001|
|RD0000011 |10.51     |6.7             |70.417            |
|RD0000012 |1.49      |6.4             |9.536             |
|RD0000013 |15.75     |6.7             |105.525           |
|RD0000014 |15.96     |8.2             |130.87199999999999|
|RD0000015 |6.06      |6.4             |

### 11 — Ticket count and average maintenance cost by site_type and severity

In [16]:
maintenance_meter_df = maintenance_df.join(meter_df, "meter_id", "left")
exercise_11_df = maintenance_meter_df.groupBy("site_type", "severity").agg(F.count("*").alias("ticket_count"), F.avg("maintenance_cost_inr").alias("average_maintenance_cost_inr")).orderBy("site_type", "severity")
exercise_11_df.show(truncate=False)

+---------------+--------+------------+----------------------------+
|site_type      |severity|ticket_count|average_maintenance_cost_inr|
+---------------+--------+------------+----------------------------+
|Commercial     |Critical|37          |13424.726216216219          |
|Commercial     |High    |130         |12293.073793103451          |
|Commercial     |Low     |166         |12223.517692307689          |
|Commercial     |Medium  |171         |13706.600886075952          |
|Industrial     |Critical|17          |12296.198000000002          |
|Industrial     |High    |63          |12838.869344262299          |
|Industrial     |Low     |66          |12717.52508474576           |
|Industrial     |Medium  |106         |12043.898526315788          |
|Public Facility|Critical|15          |8986.843571428572           |
|Public Facility|High    |48          |12905.279347826086          |
|Public Facility|Low     |60          |11195.929464285715          |
|Public Facility|Medium  |97      

### 12 — Average voltage and five highest outage substations

In [ ]:
exercise_12_df = readings_meter_substation_df.groupBy("substation_id").agg(F.avg("voltage_v").alias("average_voltage_v"), F.sum("outage_minutes").alias("total_outage_minutes")).orderBy(F.desc("total_outage_minutes")).limit(5)
exercise_12_df.show(truncate=False)

### 13 — Valid, Estimated and Rejected counts by source_type

In [17]:
exercise_13_df = readings_df.groupBy("source_type").agg(
    F.sum((F.col("reading_status") == "Valid").cast("long")).alias("valid_count"),
    F.sum((F.col("reading_status") == "Estimated").cast("long")).alias("estimated_count"),
    F.sum((F.col("reading_status") == "Rejected").cast("long")).alias("rejected_count"),
    F.count("*").alias("reading_count")).orderBy(F.desc("reading_count"))
exercise_13_df.show(truncate=False)

+-----------+-----------+---------------+--------------+-------------+
|source_type|valid_count|estimated_count|rejected_count|reading_count|
+-----------+-----------+---------------+--------------+-------------+
|SmartMeter |65521      |10443          |3202          |79166        |
|Manual     |8965       |1405           |464           |10834        |
+-----------+-----------+---------------+--------------+-------------+



## Section 02 — Advanced DataFrame Operations

### 14 — Monthly Valid energy by region

In [18]:
timestamp_readings_df = readings_df.withColumn("reading_timestamp", F.to_timestamp("reading_timestamp"))
region_readings_df = timestamp_readings_df.join(meter_df, "meter_id", "left").join(substation_df.select("substation_id", "region"), "substation_id", "left")
exercise_14_df = region_readings_df.filter(F.col("reading_status") == "Valid").withColumn("year", F.year("reading_timestamp")).withColumn("month", F.month("reading_timestamp")).groupBy("region", "year", "month").agg(F.sum("energy_kwh").alias("monthly_valid_energy_kwh")).orderBy("region", "year", "month")
exercise_14_df.show(truncate=False)

+-------+----+-----+------------------------+
|region |year|month|monthly_valid_energy_kwh|
+-------+----+-----+------------------------+
|Central|2026|1    |31475.57999999999       |
|Central|2026|2    |28161.410000000007      |
|Central|2026|3    |29805.85                |
|Central|2026|4    |29475.339999999982      |
|Central|2026|5    |30048.879999999986      |
|Central|2026|6    |27536.84999999999       |
|North  |2026|1    |48849.309999999925      |
|North  |2026|2    |45444.249999999985      |
|North  |2026|3    |51094.61000000006       |
|North  |2026|4    |43451.02000000005       |
|North  |2026|5    |48796.93000000004       |
|North  |2026|6    |47072.32999999992       |
|South  |2026|1    |97547.38999999987       |
|South  |2026|2    |90864.80000000006       |
|South  |2026|3    |102258.77000000005      |
|South  |2026|4    |97266.94000000005       |
|South  |2026|5    |101204.8699999999       |
|South  |2026|6    |97699.97000000002       |
|West   |2026|1    |59967.56999999

### 15 — Top three meters by Valid energy per substation

In [19]:
meter_energy_df = readings_meter_df.filter(F.col("reading_status") == "Valid").groupBy("substation_id", "meter_id").agg(F.sum("energy_kwh").alias("total_valid_energy_kwh"))
w15 = Window.partitionBy("substation_id").orderBy(F.desc("total_valid_energy_kwh"))
exercise_15_df = meter_energy_df.withColumn("meter_rank", F.dense_rank().over(w15)).filter(F.col("meter_rank") <= 3).orderBy("substation_id", "meter_rank")
exercise_15_df.show(truncate=False)

+-------------+--------+----------------------+----------+
|substation_id|meter_id|total_valid_energy_kwh|meter_rank|
+-------------+--------+----------------------+----------+
|SUB01        |M0130   |9364.98               |1         |
|SUB01        |M0110   |9259.750000000002     |2         |
|SUB01        |M0068   |8832.580000000005     |3         |
|SUB02        |M0071   |29242.04              |1         |
|SUB02        |M0145   |28641.319999999992    |2         |
|SUB02        |M0133   |7060.140000000003     |3         |
|SUB03        |M0096   |9632.740000000003     |1         |
|SUB03        |M0080   |9525.940000000002     |2         |
|SUB03        |M0073   |9369.789999999995     |3         |
|SUB04        |M0038   |29505.88              |1         |
|SUB04        |M0099   |29216.319999999996    |2         |
|SUB04        |M0098   |28613.460000000014    |3         |
|SUB05        |M0094   |9659.639999999996     |1         |
|SUB05        |M0050   |9570.859999999997     |2        

### 16 — Cumulative Valid energy per meter

In [20]:
valid_timestamp_df = timestamp_readings_df.filter(F.col("reading_status") == "Valid")
w16 = Window.partitionBy("meter_id").orderBy("reading_timestamp", "reading_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
exercise_16_df = valid_timestamp_df.withColumn("cumulative_valid_energy_kwh", F.sum("energy_kwh").over(w16))
exercise_16_df.select("meter_id", "reading_timestamp", "reading_id", "energy_kwh", "cumulative_valid_energy_kwh").show(20, truncate=False)

+--------+-------------------+----------+----------+---------------------------+
|meter_id|reading_timestamp  |reading_id|energy_kwh|cumulative_valid_energy_kwh|
+--------+-------------------+----------+----------+---------------------------+
|M0001   |2026-01-01 01:15:00|RD0083587 |22.27     |22.27                      |
|M0001   |2026-01-01 03:30:00|RD0086275 |4.05      |26.32                      |
|M0001   |2026-01-01 09:45:00|RD0088204 |11.28     |37.6                       |
|M0001   |2026-01-02 04:45:00|RD0005149 |11.76     |49.36                      |
|M0001   |2026-01-02 08:15:00|RD0035530 |10.06     |59.42                      |
|M0001   |2026-01-02 18:30:00|RD0070104 |15.83     |75.25                      |
|M0001   |2026-01-03 00:30:00|RD0006677 |17.06     |92.31                      |
|M0001   |2026-01-03 06:00:00|RD0021042 |17.14     |109.45                     |
|M0001   |2026-01-03 08:30:00|RD0032205 |20.46     |129.91                     |
|M0001   |2026-01-03 18:15:0

### 17 — Union both reading datasets

In [21]:
combined_readings_df = readings_df.unionByName(extra_readings_df, allowMissingColumns=True)
print("Main:", readings_df.count(), "Extra:", extra_readings_df.count(), "After union:", combined_readings_df.count())

Main: 90000 Extra: 12000 After union: 102000


### 18 — Find duplicate IDs and deduplicate

In [22]:
repeated_ids_df = combined_readings_df.groupBy("reading_id").count().filter(F.col("count") > 1)
repeated_ids_df.show(truncate=False)
unique_readings_df = combined_readings_df.dropDuplicates(["reading_id"])
print("Before:", combined_readings_df.count(), "After:", unique_readings_df.count(), "Removed:", combined_readings_df.count() - unique_readings_df.count())

+----------+-----+
|reading_id|count|
+----------+-----+
|RD0025848 |2    |
|RD0045533 |2    |
|RD0064921 |2    |
|RD0056575 |2    |
|RD0058815 |2    |
|RD0022131 |2    |
|RD0029598 |2    |
|RD0045710 |2    |
|RD0030877 |2    |
|RD0002679 |2    |
|RD0011169 |2    |
|RD0028736 |2    |
|RD0030669 |2    |
|RD0061933 |2    |
|RD0015904 |2    |
|RD0008188 |2    |
|RD0037899 |2    |
|RD0000004 |2    |
|RD0025628 |2    |
|RD0048911 |2    |
+----------+-----+
only showing top 20 rows
Before: 102000 After: 101300 Removed: 700


### 19 — Lazy pipeline and explain

In [23]:
exercise_19_df = readings_df.filter(F.col("reading_status") == "Valid").withColumn("energy_value_kwh", F.col("energy_kwh")).groupBy("meter_id").agg(F.sum("energy_value_kwh").alias("total_valid_energy_kwh")).orderBy(F.desc("total_valid_energy_kwh"))
exercise_19_df.explain()
exercise_19_df.show(10, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_valid_energy_kwh#1035 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_valid_energy_kwh#1035 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2440]
      +- HashAggregate(keys=[meter_id#18], functions=[sum(energy_value_kwh#1034)])
         +- Exchange hashpartitioning(meter_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=2437]
            +- HashAggregate(keys=[meter_id#18], functions=[partial_sum(energy_value_kwh#1034)])
               +- Project [meter_id#18, energy_kwh#20 AS energy_value_kwh#1034]
                  +- Filter (isnotnull(reading_status#24) AND (reading_status#24 = Valid))
                     +- FileScan csv [meter_id#18,energy_kwh#20,reading_status#24] Batched: false, DataFilters: [isnotnull(reading_status#24), (reading_status#24 = Valid)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(readi

### 20 — Cache Valid readings and time count twice

In [24]:
valid_cached_df = readings_df.filter(F.col("reading_status") == "Valid").cache()
start = time.time(); valid_count_1 = valid_cached_df.count(); elapsed_1 = time.time() - start
start = time.time(); valid_count_2 = valid_cached_df.count(); elapsed_2 = time.time() - start
print("is_cached:", valid_cached_df.is_cached)
print("Counts:", valid_count_1, valid_count_2)
print(f"Elapsed: {elapsed_1:.3f}s and {elapsed_2:.3f}s")
valid_cached_df.unpersist()

is_cached: True
Counts: 74486 74486
Elapsed: 2.016s and 0.128s


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

### 21 — Persist outage readings, run two actions, unpersist

In [25]:
outage_df = readings_df.filter(F.col("outage_minutes") > 0).persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", outage_df.count())
print("Sample rows:", len(outage_df.take(10)))
print("Storage level:", outage_df.storageLevel)
outage_df.unpersist()

Count: 7653
Sample rows: 10
Storage level: Disk Memory Serialized 1x Replicated


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

### 22 — Compare repeated aggregation with and without caching

In [26]:
def run_meter_aggregation(df):
    start = time.time()
    groups = df.groupBy("meter_id").agg(F.sum("energy_kwh").alias("total_energy")).count()
    return groups, time.time() - start
uncached_1, t_uncached_1 = run_meter_aggregation(readings_df)
uncached_2, t_uncached_2 = run_meter_aggregation(readings_df)
cached_readings_df = readings_df.cache()
cached_readings_df.count()
cached_1, t_cached_1 = run_meter_aggregation(cached_readings_df)
cached_2, t_cached_2 = run_meter_aggregation(cached_readings_df)
print("Uncached:", uncached_1, uncached_2, f"{t_uncached_1:.3f}s / {t_uncached_2:.3f}s")
print("Cached:", cached_1, cached_2, f"{t_cached_1:.3f}s / {t_cached_2:.3f}s")
cached_readings_df.unpersist()

Uncached: 150 150 0.581s / 0.839s
Cached: 150 150 0.442s / 0.427s


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

### 23 — Eight partitions and rows per partition

In [ ]:
exercise_23_df = readings_df.repartition(8).withColumn("partition_id", F.spark_partition_id())
exercise_23_df.groupBy("partition_id").count().orderBy("partition_id").show()

### 24 — Compare coalesce(3) and repartition(3)

In [27]:
readings_12_df = readings_df.repartition(12)
coalesced_3_df = readings_12_df.coalesce(3)
repartitioned_3_df = readings_12_df.repartition(3)
print("12:", readings_12_df.rdd.getNumPartitions(), "coalesce:", coalesced_3_df.rdd.getNumPartitions(), "repartition:", repartitioned_3_df.rdd.getNumPartitions())
print("coalesce plan:"); coalesced_3_df.explain()
print("repartition plan:"); repartitioned_3_df.explain()

12: 12 coalesce: 3 repartition: 3
coalesce plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3149]
            +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3149]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_m

### 25 — Join cleaned readings to meter/substation and write Parquet

In [28]:
clean_readings_df = readings_df.na.drop(subset=["energy_kwh", "voltage_v"])
enriched_readings_df = clean_readings_df.join(meter_df, "meter_id", "left").join(substation_df, "substation_id", "left").repartition(4)
energy_parquet_path = "/content/enriched_energy_readings_parquet"
enriched_readings_df.write.mode("overwrite").parquet(energy_parquet_path)
print("Saved:", energy_parquet_path)
print([p.name for p in Path(energy_parquet_path).iterdir()])

Saved: /content/enriched_energy_readings_parquet
['part-00002-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet', 'part-00001-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet', '.part-00002-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet.crc', 'part-00003-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet', 'part-00000-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet', '.part-00001-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet.crc', '._SUCCESS.crc', '.part-00000-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet.crc', '.part-00003-6f017668-279c-4cca-8113-c571fc540bb0-c000.snappy.parquet.crc', '_SUCCESS']


## Section 03 — Integrated Assessment: Final Capstone

In [29]:
capstone_combined_df = readings_df.unionByName(extra_readings_df, allowMissingColumns=True)
capstone_unique_df = capstone_combined_df.dropDuplicates(["reading_id"])
print("Combined:", capstone_combined_df.count(), "Unique:", capstone_unique_df.count())

Combined: 102000 Unique: 101300


In [30]:
capstone_clean_df = capstone_unique_df.withColumn("reading_timestamp", F.to_timestamp("reading_timestamp")).withColumn(
    "month", F.date_format("reading_timestamp", "yyyy-MM")
).withColumn("outage_minutes_clean", F.coalesce(F.col("outage_minutes"), F.lit(0))).filter(
    (F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull()
)

In [31]:
capstone_joined_df = capstone_clean_df.join(meter_df, "meter_id", "left").join(
    substation_df, "substation_id", "left"
).join(tariff_df, "tariff_id", "left")

In [32]:
capstone_calculated_df = capstone_joined_df.withColumn(
    "energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr")
).withColumn("has_outage", F.when(F.col("outage_minutes_clean") > 0, 1).otherwise(0))

In [33]:
capstone_report_df = capstone_calculated_df.groupBy("month", "region", "site_type").agg(
    F.sum("energy_kwh").alias("total_billable_energy_kwh"),
    F.sum("energy_charge_inr").alias("total_energy_charges_inr"),
    F.sum("outage_minutes_clean").alias("total_outage_minutes"),
    F.count("*").alias("reading_count"),
    F.sum("has_outage").alias("readings_with_outage")
).withColumn("outage_reading_pct", F.col("readings_with_outage") / F.col("reading_count") * 100)

In [34]:
rank_window = Window.partitionBy("month", "region").orderBy(F.desc("total_energy_charges_inr"))
capstone_final_df = capstone_report_df.withColumn("site_type_rank", F.dense_rank().over(rank_window)).cache()

In [35]:
capstone_final_df.orderBy("month", "region", "site_type_rank").show(50, truncate=False)
print("Report rows:", capstone_final_df.count())
print("Total billable kWh:", capstone_final_df.agg(F.sum("total_billable_energy_kwh")).first()[0])

+-------+-------+---------------+-------------------------+------------------------+--------------------+-------------+--------------------+------------------+--------------+
|month  |region |site_type      |total_billable_energy_kwh|total_energy_charges_inr|total_outage_minutes|reading_count|readings_with_outage|outage_reading_pct|site_type_rank|
+-------+-------+---------------+-------------------------+------------------------+--------------------+-------------+--------------------+------------------+--------------+
|2026-01|Central|Industrial     |24871.22                 |226328.102              |4183                |418          |47                  |11.24401913875598 |1             |
|2026-01|Central|Commercial     |10649.029999999995       |87322.04599999999       |4876                |567          |55                  |9.700176366843033 |2             |
|2026-01|Central|Residential    |814.03                   |4636.908000000001       |1768                |224          |19    

In [36]:
capstone_path = "/content/energy_consumption_reliability_report"
capstone_final_df.repartition(2).write.mode("overwrite").parquet(capstone_path)
print("Parquet report:", capstone_path)
print([p.name for p in Path(capstone_path).iterdir()])
capstone_final_df.unpersist()

Parquet report: /content/energy_consumption_reliability_report
['part-00000-063d5857-3396-47ac-b713-522569fe19f2-c000.snappy.parquet', '.part-00000-063d5857-3396-47ac-b713-522569fe19f2-c000.snappy.parquet.crc', '._SUCCESS.crc', '.part-00001-063d5857-3396-47ac-b713-522569fe19f2-c000.snappy.parquet.crc', 'part-00001-063d5857-3396-47ac-b713-522569fe19f2-c000.snappy.parquet', '_SUCCESS']


DataFrame[month: string, region: string, site_type: string, total_billable_energy_kwh: double, total_energy_charges_inr: double, total_outage_minutes: bigint, reading_count: bigint, readings_with_outage: bigint, outage_reading_pct: double, site_type_rank: int]